# Deep Learning: Transformers

In Lesson 16, we solved the Vanishing Gradient problem using the LSTM. By introducing a "Cell State" conveyor belt, our Neural Network could finally remember events that happened 30 days ago.

But LSTMs have a fatal architectural flaw: **They are strictly sequential.** They must read Day 1, then Day 2, then Day 3, in perfect order. Because of this, they are incredibly slow to train, and even with the Cell State, they still struggle to connect Day 1 to Day 1,000.

In 2017, Google researchers published a paper called *"Attention Is All You Need"*. They proposed an architecture that threw away the sequential loop entirely. It looks at the entire timeline simultaneously, drawing direct mathematical connections between any two points in time, instantly. This architecture is the **Transformer**, and it powers everything from ChatGPT to modern algorithmic trading.

To apply a Transformer to Time Series forecasting, we must translate its core mechanism—**Self-Attention**—from the world of Natural Language Processing (words) into the world of continuous numerical sequences (lags and horizons).

Let's set up our Python environment using TensorFlow/Keras to build a modern Attention mechanism.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LayerNormalization, MultiHeadAttention, Dropout, GlobalAveragePooling1D
from tensorflow.keras.models import Model

# Set professional visualization styling
sns.set_theme(style="whitegrid")

print("✅ Transformer & Attention Architecture Environment Ready.")

✅ Transformer & Attention Architecture Environment Ready.


# 1. The Mathematics of Self-Attention

In an LSTM, if the algorithm wants to know how the sales on Black Friday (Day 50) relate to New Year's Day (Day 80), the data must pass through 30 intermediate days of hidden states.

**Self-Attention** creates a direct, $O(1)$ mathematical bridge between Day 50 and Day 80, completely bypassing the days in between. It calculates a massive correlation matrix, asking: *"For every single time step in my sequence, how much attention should I pay to every other time step?"*

It does this using three learned matrices: **Queries ($Q$)**, **Keys ($K$)**, and **Values ($V$)**.

* **Query ($Q$)**: What is Day 80 looking for? (e.g., "I need to know about recent massive spikes.")
* **Key ($K$)**: What does Day 50 have? (e.g., "I am a massive spike.")
* **Value ($V$)**: The actual numerical payload of Day 50.

### The Attention Equation

The network takes the dot product of the Queries and the Keys to calculate a raw "Match Score." It scales this score down by the square root of the matrix dimension ($d_k$) to prevent exploding gradients, applies a Softmax function to turn the scores into percentages (summing to $1.0$), and multiplies those percentages by the Values.

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

# 2. The Positional Encoding Problem

Because the Transformer calculates the $Q K^T$ matrix for all time steps simultaneously, it processes the entire sequence in parallel.

This creates a paradox: **The Transformer has no concept of Time.** If you scramble the order of the days, a raw Attention mechanism will output the exact same result. It sees the data as a "bag of numbers," not a sequence.

To fix this, we must explicitly inject the clock back into the data before feeding it to the Attention layer. We do this using **Positional Encoding**.
Just like we used Trigonometry in Lesson 06 to encode the months of the year, Transformers use overlapping Sine and Cosine waves of different frequencies to stamp a unique, permanent mathematical "time signature" onto every single row of data.

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

# 3. Enterprise Time Series Transformers

Standard NLP Transformers (like the ones powering GPT-4) are designed to output discrete words (Classification). Time Series requires continuous numbers (Regression) and the handling of Exogenous variables.

Enterprise Data Scientists use specialized variants:

* **Temporal Fusion Transformer (TFT)**: Developed by Google, TFT is the absolute gold standard for retail forecasting. It natively separates static variables (Store Location) from known future variables (Upcoming Holidays) from unknown future variables (Sales).
* **N-BEATS**: Developed by Element AI, this architecture specifically focuses on highly interpretable decomposition, breaking the forecast directly into distinct Trend and Seasonality outputs that humans can read.

# 4. Building a Time Series Transformer Block in Code

Building a full TFT from scratch requires thousands of lines of code. Instead, let's architect a custom **Transformer Encoder Block** in Keras. We will feed it a 3D Time Series tensor, apply Multi-Head Attention, and use it to forecast a complex sequence.

In [4]:
# 1. Architect the Transformer Block
def build_transformer_model(look_back, num_features, head_size, num_heads, ff_dim, dropout=0.1):
    """Builds a Time Series Transformer using Keras Functional API."""
    
    # Input Layer: (Batch_Size, Time_Steps, Features)
    inputs = Input(shape=(look_back, num_features))
    
    # --- Attention Mechanism ---
    # MultiHeadAttention splits the Q, K, V matrices into multiple "heads", 
    # allowing the network to look for different types of patterns simultaneously.
    attention_output = MultiHeadAttention(
        key_dim=head_size, num_heads=num_heads, dropout=dropout
    )(inputs, inputs)
    
    # Skip Connection & Layer Normalization
    # We add the original input back to the attention output (Residual Connection) to prevent gradient loss
    x = LayerNormalization(epsilon=1e-6)(inputs + attention_output)
    
    # --- Feed Forward Network ---
    # A standard dense network to process the contextualized outputs
    ffn_output = Dense(ff_dim, activation="relu")(x)
    ffn_output = Dropout(dropout)(ffn_output)
    ffn_output = Dense(num_features)(ffn_output)
    
    # Second Skip Connection & Normalization
    x = LayerNormalization(epsilon=1e-6)(x + ffn_output)
    
    # --- Final Output Layer ---
    # Pool the sequence down to a flat vector, and predict 1 step into the future
    x = GlobalAveragePooling1D()(x)
    x = Dropout(dropout)(x)
    outputs = Dense(1, activation="linear")(x)
    
    return Model(inputs, outputs)

# 2. Instantiate the Enterprise Architecture
look_back_window = 30 # Look at the past 30 days
num_features = 1      # Univariate series

model = build_transformer_model(
    look_back=look_back_window,
    num_features=num_features,
    head_size=64,   # Dimensions of the Q, K, V matrices
    num_heads=4,    # 4 separate "brains" looking at the sequence
    ff_dim=128      # Neurons in the dense layer
)

model.compile(loss="mse", optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4))
print("🚨 Transformer Architecture Compiled Successfully.")
model.summary()

# NOTE: To train this, you would use the exact same tabularization 
# and 3D tensor creation function (create_3d_dataset) from Lesson 16!

🚨 Transformer Architecture Compiled Successfully.


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 30, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 30, 1)     │      1,793 │ input_layer_1[0]… │
│ (MultiHeadAttentio… │                   │            │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 30, 1)     │          0 │ input_layer_1[0]… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 30, 1)     │          2 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 30, 128)   │        256 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 30, 128)   │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 30, 1)     │        129 │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 30, 1)     │          0 │ layer_normalizat… │
│                     │                   │            │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 30, 1)     │          2 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1)         │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 1)         │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1)         │          2 │ dropout_5[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,184 (8.53 KB)

 Trainable params: 2,184 (8.53 KB)

 Non-trainable params: 0 (0.00 B)

## Real-World Use Case or Analogy:

Think of the difference between an LSTM and a Transformer like **A Detective Solving a Case**:

* **The LSTM (The Reader)**: The detective is handed a 1,000-page case file. They must read it linearly, from page 1 to page 1,000. By the time they reach the end, their memory of the clues on page 12 is fuzzy and degraded. If you ask them a question, they have to re-read the whole book.
* **The Transformer (The Corkboard)**: The detective takes all 1,000 pages, tears them out, and pins them all to a massive wall simultaneously. They use red string (**Self-Attention**) to draw a direct line between a clue on page 12 and a suspect on page 980. They do not have to read the pages in between; they just follow the red string.
* *Query*: "Who matches the description on page 980?"
* *Key*: Page 12 says, "I have a physical description."
* *Dot Product*: The detective mathematically links them, instantly bridging the gap of time.